# 05 · Work IQ MCP 로 데이터 가져오기

**Work IQ MCP 파이프라인 1/2** — Agent365(메일/Teams 개별 서버) 대신 **Work IQ MCP**
(단일 엔드포인트 + 범용 도구)에 연결해, 에이전트/LLM 없이 **도구를 직접 호출**해 데이터를
가져옵니다. `03_fetch_data`와 같은 흐름이지만, 도구가 소스별 이름(`SearchMessages`)이 아니라
**Graph 경로 기반 범용 도구**(`fetch`, `search_paths`, `ask`)라는 점이 다릅니다.

> ℹ️ Work IQ는 **회사·학교(Entra) 계정 + 관리자 동의 + EULA**가 필요합니다. 최초 실행 시
> 브라우저 로그인/동의가 한 번 뜨고, 이후 `.workiq.token_cache.json`으로 재사용됩니다.

## 셋업 (설정 + 인증)

In [ ]:
# ============================================================
# 셋업 — 이 셀 하나로 설정·인증 준비 (pipeline/*.py 없이 독립 실행)
# Work IQ MCP: 단일 엔드포인트 + 범용 10개 도구(fetch/ask/search_paths/...).
# 기본값은 Microsoft 공개 클라이언트라 별도 앱 등록이 필요 없습니다.
# (회사·학교 계정 + 관리자 동의 + EULA 필요, 개인 계정 불가)
# ============================================================
import os, json, asyncio
from pathlib import Path
from contextlib import AsyncExitStack

import msal
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client, create_mcp_http_client

# 1) 프로젝트 루트를 찾아 .env 로드 (.env.example/requirements.txt가 있는 폴더)
def find_project_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / '.env.example').is_file() or (base / 'requirements.txt').is_file():
            return base
    return Path.cwd()

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env')

# 2) Work IQ MCP 설정 (모두 .env로 덮어쓸 수 있음)
#    - 엔드포인트/스코프/공개 클라이언트 기본값은 Work IQ 공식 값입니다.
TENANT_ID     = (os.environ.get('TENANT_ID') or '').strip() or 'organizations'
WORKIQ_URL    = (os.environ.get('WORKIQ_MCP_SERVER_URL') or '').strip() or 'https://workiq.svc.cloud.microsoft/mcp'
CLIENT_ID     = (os.environ.get('WORKIQ_CLIENT_ID') or '').strip() or 'ba081686-5d24-4bc6-a0d6-d034ecffed87'
SCOPE         = (os.environ.get('WORKIQ_SCOPE') or '').strip() or 'fdcc1f02-fc51-4226-8753-f668596af7f7/WorkIQAgent.Ask'
AUTH_MODE     = (os.environ.get('WORKIQ_AUTH_MODE') or 'interactive').strip()   # interactive(기본) | device_code
REDIRECT_PORT = int((os.environ.get('WORKIQ_REDIRECT_PORT') or '12798').strip())
AUTHORITY     = f'https://login.microsoftonline.com/{TENANT_ID}'
SCOPES        = [SCOPE]
TOKEN_CACHE   = PROJECT_ROOT / ((os.environ.get('WORKIQ_TOKEN_CACHE_PATH') or '').strip() or '.workiq.token_cache.json')

# 3) MSAL 인증 — Work IQ 위임(delegated) 토큰. 최초 1회 로그인/동의 후 캐시 재사용.
#    interactive : 로컬 브라우저 로그인(루프백). 로컬 Jupyter 기본값.
#    device_code : 원격/헤드리스 커널용. https://microsoft.com/devicelogin 에 코드 입력.
_cache = msal.SerializableTokenCache()
if TOKEN_CACHE.exists():
    _cache.deserialize(TOKEN_CACHE.read_text(encoding='utf-8'))

def _save_cache():
    if _cache.has_state_changed:
        TOKEN_CACHE.write_text(_cache.serialize(), encoding='utf-8')

def ensure_token() -> str:
    """Work IQ access token 확보 (캐시 우선, 없으면 로그인)."""
    assert CLIENT_ID, 'WORKIQ_CLIENT_ID가 필요합니다 (기본 공개 클라이언트 사용 가능).'
    app = msal.PublicClientApplication(CLIENT_ID, authority=AUTHORITY, token_cache=_cache)
    accounts = app.get_accounts()
    res = app.acquire_token_silent(SCOPES, account=accounts[0]) if accounts else None
    if not (res and 'access_token' in res):
        if AUTH_MODE == 'device_code':                 # 원격/헤드리스
            flow = app.initiate_device_flow(scopes=SCOPES)
            print(flow['message'])                      # 인증 URL + 코드 안내
            res = app.acquire_token_by_device_flow(flow)
        else:                                           # 기본: 브라우저 로그인
            res = app.acquire_token_interactive(scopes=SCOPES, port=REDIRECT_PORT)
    _save_cache()
    if not res or 'access_token' not in res:
        raise RuntimeError((res or {}).get('error_description', 'access token 획득 실패'))
    return res['access_token']

print('PROJECT_ROOT:', PROJECT_ROOT)
print('WORKIQ_URL  :', WORKIQ_URL)
print('AUTHORITY   :', AUTHORITY, '| CLIENT_ID set:', bool(CLIENT_ID), '| AUTH_MODE:', AUTH_MODE)
print('SCOPE       :', SCOPE)
print('TOKEN_CACHE :', TOKEN_CACHE.name)

In [ ]:
# --- MCP 헬퍼: 단일 Work IQ 엔드포인트. 연결은 호출마다 열고 항상 닫는다 ---
async def _with_session(token, fn):
    # 최신 MCP SDK: 인증 헤더는 httpx 클라이언트에 담아 streamable_http_client에 전달
    async with create_mcp_http_client(headers={'Authorization': f'Bearer {token}'}) as http_client:
        async with streamable_http_client(WORKIQ_URL, http_client=http_client) as (r, w, _):
            async with ClientSession(r, w) as s:       # MCP 세션 초기화(handshake)
                await s.initialize()
                return await fn(s)

async def list_tools(token):
    res = await _with_session(token, lambda s: s.list_tools())
    return list(res.tools or [])

async def call_tool(token, name, args=None):
    return await _with_session(token, lambda s: s.call_tool(name, arguments=args or {}))

def tool_schema(tool):
    sc = getattr(tool, 'inputSchema', None) or {'type': 'object', 'properties': {}}
    return sc.model_dump() if hasattr(sc, 'model_dump') else sc

def tool_payload(result):
    """Work IQ 결과를 dict로: structuredContent 우선, 없으면 text 블록을 JSON 파싱."""
    sc = getattr(result, 'structuredContent', None)
    if isinstance(sc, dict) and sc:
        return sc
    for b in (getattr(result, 'content', None) or []):
        if getattr(b, 'type', None) == 'text' and getattr(b, 'text', ''):
            try:
                return json.loads(b.text)
            except Exception:
                pass
    return None

def content_to_text(result) -> str:
    """content 블록을 평탄화 (\\uXXXX 이스케이프 JSON은 한글로 복원)."""
    parts = []
    for b in (getattr(result, 'content', None) or []):
        if getattr(b, 'type', None) == 'text':
            t = getattr(b, 'text', '')
            try:
                t = json.dumps(json.loads(t), ensure_ascii=False, indent=2)
            except Exception:
                pass
            parts.append(t)
        else:
            data = b.model_dump() if hasattr(b, 'model_dump') else b
            parts.append('```json\n' + json.dumps(data, ensure_ascii=False, indent=2, default=str) + '\n```')
    text = '\n'.join(parts)
    return f'ERROR: {text}' if getattr(result, 'isError', False) else text

def readable(result) -> str:
    """사람이 보기 좋게: ask→response, search_paths→paths, fetch→results[].data(.value), 그 외 JSON."""
    if getattr(result, 'isError', False):
        return 'ERROR: ' + content_to_text(result)
    p = tool_payload(result)
    if isinstance(p, dict):
        if 'response' in p:                                  # ask
            return p.get('response') or '(빈 응답)'
        if 'paths' in p:                                     # search_paths
            return '\n'.join(f" - {x.get('path')}  [{', '.join(x.get('operations', []))}]"
                             for x in (p.get('paths') or []))
        if 'results' in p:                                   # fetch
            out = []
            for r in (p.get('results') or []):
                data = r.get('data') if isinstance(r, dict) else r
                sc = r.get('statusCode') if isinstance(r, dict) else None
                body = data.get('value') if isinstance(data, dict) and 'value' in data else data
                out.append((f'[statusCode={sc}]\n' if sc else '') +
                           json.dumps(body, ensure_ascii=False, indent=2))
            return '\n'.join(out)
        return json.dumps(p, ensure_ascii=False, indent=2)
    return content_to_text(result)

## 1. 토큰 + 툴 목록
Work IQ는 **10개 범용 도구**를 노출합니다: `fetch`/`create_entity`/`update_entity`/
`delete_entity`/`do_action`/`call_function`(엔티티), `ask`/`list_agents`(Copilot),
`get_schema`/`search_paths`(스키마). 이 노트북은 **읽기 도구만** 사용합니다.

In [ ]:
token = ensure_token()
tools = await list_tools(token)
print('연결된 Work IQ 도구:', ', '.join(t.name for t in tools))

## 2. `search_paths` 로 경로 탐색
Work IQ의 설계 원칙은 *introspection over enumeration* — 수천 개 타입을 미리 넣지 않고
**필요할 때 경로를 검색**합니다. 필터(접두사/정규식)로 사용 가능한 Graph 경로를 찾아봅니다.

In [ ]:
for f in ['messages', '.*chats.*', '.*calendar.*']:
    print(f'\n===== search_paths(filter={f!r}) =====')
    try:
        res = await call_tool(token, 'search_paths', {'filter': f})
        print(readable(res)[:1500] or '(빈 응답)')
    except Exception as exc:
        print('  실패:', exc)

## 3. `fetch` 로 직접 읽기
`fetch`는 **상대 Graph 경로**를 그대로 읽습니다(원시 조회, 의미검색 아님).
`$select`/`$top`으로 범위를 좁히고, `/me/chats`는 **채팅 목록(메타데이터)**이라 메시지는
`/me/chats/{id}/messages`로 한 단계 더 들어가 읽습니다. (컬렉션 기본 $top=25, 최대 100)

In [ ]:
# 메일: 최근 메일을 필요한 필드만
mail_url = '/me/messages?$select=id,subject,from,receivedDateTime,bodyPreview&$top=10'
print('===== fetch', mail_url, '=====')
try:
    res = await call_tool(token, 'fetch', {'entityUrls': [mail_url]})
    print(readable(res)[:3000] or '(빈 응답)')
except Exception as exc:
    print('  실패:', exc)

# Teams: 채팅 목록(메타데이터) → 첫 채팅의 메시지
print('\n===== fetch /me/chats (목록) =====')
chat_id = None
try:
    res = await call_tool(token, 'fetch', {'entityUrls': ['/me/chats?$top=10']})
    p = tool_payload(res) or {}
    val = (((p.get('results') or [{}])[0]).get('data') or {}).get('value') or []
    for c in val[:10]:
        print(f"  - {c.get('id')}  topic={c.get('topic')!r} type={c.get('chatType')}")
    chat_id = val[0]['id'] if val else None
except Exception as exc:
    print('  실패:', exc)

if chat_id:
    msg_url = f'/me/chats/{chat_id}/messages?$top=10'   # 채팅 메시지는 최대 10개
    print('\n===== fetch', msg_url, '=====')
    try:
        res = await call_tool(token, 'fetch', {'entityUrls': [msg_url]})
        print(readable(res)[:3000] or '(빈 응답)')
    except Exception as exc:
        print('  실패:', exc)

## 4. `ask` 로 자연어 조회 (02 샘플 되읽기)
`ask`는 **M365 Copilot**을 호출해 메일·Teams를 가로질러 의미검색/추론합니다.
`02_seed_sample_data`에서 넣은 지식을 자연어로 되읽어봅니다.

> ⚠️ 의미 색인은 **지연**될 수 있어 방금 넣은 샘플이 즉시 안 잡힐 수 있습니다. 그럴 땐
> 위 **3번 `fetch`** 가 더 즉각적입니다.

In [ ]:
ASKS = [
    'Postgres 커넥션 풀 튜닝과 k8s CrashLoopBackOff(파드) 관련 최근 이메일 내용을 요약해줘.',
    'GitHub Actions 캐시 히트율과 사내 로그 표준(JSON, trace_id) 관련 Teams 메시지를 찾아 요약해줘.',
]
for q in ASKS:
    print(f'\n===== ask: {q} =====')
    try:
        res = await call_tool(token, 'ask', {'question': q, 'timeZone': 'Asia/Seoul'})
        print(readable(res)[:3000] or '(빈 응답)')
    except Exception as exc:
        print('  실패:', exc)

✅ Work IQ MCP로 데이터를 직접 가져올 수 있음을 확인했습니다. 다음은 이 과정을 자연어로
취합해 위키 Markdown을 만드는 **06_workiq_aggregate_to_md** 입니다.